<a href="https://colab.research.google.com/github/faeze-bnt/Attack-project/blob/colab/Attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install larq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 1.4 MB/s eta 0:00:00


In [5]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import SGD

import tensorflow as tf

import os

# Enable XLA devices
tf.config.optimizer.set_jit(True)

In [7]:
import tensorflow as tf
import larq as lq

In [8]:
from tensorflow.keras.models import Sequential

In [3]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

train_images = train_images.reshape((60000, 28, 28, 1))
test_images = test_images.reshape((10000, 28, 28, 1))

# Normalize pixel values to be between -1 and 1
train_images, test_images = train_images / 127.5 - 1, test_images / 127.5 - 1

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
def train(data, file_name_n, params, num_epochs=50, batch_size=128, train_temp=1, init=None):
    """
    Binarized neural network training procedure.
    """



    kwargs = dict(input_quantizer="ste_sign",
              kernel_quantizer="ste_sign",
              kernel_constraint="weight_clip")




    file_name = file_name_n+".h5"
    model = Sequential()

    # In the first layer we only quantize the weights and not the input
    model.add(lq.layers.QuantConv2D(params[0], (3, 3),
                                kernel_quantizer="ste_sign",
                                kernel_constraint="weight_clip",
                                #use_bias=False,
                                input_shape=data.train_data.shape[1:]))
    # model.add(Conv2D(params[0], (3, 3), input_shape=data.train_data.shape[1:]))

    model.add(Activation('relu'))
    model.add(Conv2D(params[1], (3, 3)))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Conv2D(params[2], (3, 3)))
    model.add(Activation('relu'))
    model.add(Conv2D(params[3], (3, 3)))
    model.add(Activation('relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(Dense(params[4]))
    model.add(Activation('relu'))
    model.add(Dropout(0.5))
    model.add(Dense(params[5]))
    model.add(Activation('relu'))
    model.add(Dense(10))

    if init != None:
        model.load_weights(init)

    def fn(correct, predicted):
        return tf.nn.softmax_cross_entropy_with_logits(labels=correct,
                                                       logits=predicted/train_temp)
    #   ts.keras.losses.CategoticalCrossentropy()

    sgd = SGD(learning_rate=0.01, momentum=0.9, nesterov=True) #decay=1e-6,

    model.compile(loss=fn,
                  optimizer=sgd,
                  metrics=['accuracy'])

    model.fit(data.train_data, data.train_labels,
              batch_size=batch_size,
              validation_data=(data.validation_data, data.validation_labels),
              epochs=num_epochs,
              shuffle=True)


    if file_name != None:
        model.save(file_name)

    return model